# ConvTran: Convolutional Transformer for Time Series Classification

This notebook demonstrates how to use the ConvTran model for time series classification tasks. We'll cover:

1. Setting up the environment
2. Preparing time series data
3. Building the ConvTran model
4. Training and evaluating the model
5. Examining model components and visualizing results

## 1. Setup

First, let's import the necessary libraries and set up our environment:

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Add the parent directory to path to allow importing from src
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# Set style for plots
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

Now, let's import the ConvTran model and related utilities:

In [ ]:
from src.models.convtran.model import ConvTran, CasualConvTran, model_factory
from src.data.loader import (
    load_dataset,
    create_dataloaders,
    TimeSeriesDataset,
    normalize_time_series,
)
from src.utils.utils import plot_confusion_matrix

## 2. Data Preparation

We'll use the UCR/UEA Archive dataset for this demonstration. If it's not available, we'll create a synthetic dataset.

In [ ]:
def create_synthetic_dataset(n_samples=200, seq_len=128, n_channels=1, n_classes=2):
 """Create a synthetic dataset for time series classification"""
 # Create arrays to hold the data
 X = np.zeros((n_samples, n_channels, seq_len))
 y = np.zeros(n_samples, dtype=int)

 # Time points
 t = np.linspace(0, 2 * np.pi, seq_len)

 # Samples per class
 samples_per_class = n_samples // n_classes

 for cls in range(n_classes):
 start_idx = cls * samples_per_class
 end_idx = (cls + 1) * samples_per_class

 # Generate patterns based on class
 if cls == 0:
 # Class 0: Sine wave with noise
 for i in range(start_idx, end_idx):
 X[i, 0, :] = np.sin(t + np.random.normal(0, 0.1)) + np.random.normal(
 0, 0.1, seq_len
 )

 elif cls == 1:
 # Class 1: Sine wave with spike and noise
 for i in range(start_idx, end_idx):
 X[i, 0, :] = np.sin(t + np.random.normal(0, 0.1))
 # Add spike
 spike_pos = np.random.randint(seq_len // 3, 2 * seq_len // 3)
 X[i, 0, spike_pos : spike_pos + 5] += 2.0
 X[i, 0, :] += np.random.normal(0, 0.1, seq_len)

 # Assign class labels
 y[start_idx:end_idx] = cls

 # Split into train and test sets (70/30)
 from sklearn.model_selection import train_test_split

 X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.3, random_state=42, stratify=y
 )

 class_names = [f"Class_{i}" for i in range(n_classes)]
 return X_train, y_train, X_test, y_test, class_names

In [ ]:
# Try to load dataset using the unified loader
dataset_name = "ECG200"
data_path = os.path.join("..", "data", "UCR", dataset_name)

try:
    # Try to load using the updated loader
    train_dataset, test_dataset = load_dataset(
        data_path=data_path,
        format_type="ucr",
        normalize=True,
        dataset_name=dataset_name,
    )

    # Extract a few samples for visualization
    sample_train_x, sample_train_y = train_dataset[0][0], train_dataset[0][1]
    X_train_shape = (
        len(train_dataset),
        sample_train_x.shape[0],
        sample_train_x.shape[1],
    )
    X_test_shape = (len(test_dataset), sample_train_x.shape[0], sample_train_x.shape[1])

    # Get unique class labels
    all_labels = [y for _, y, _ in train_dataset] + [y for _, y, _ in test_dataset]
    unique_labels = sorted(set(all_labels))
    class_names = [str(label) for label in unique_labels]

    print("Dataset loaded successfully from UCR archive")

except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Creating synthetic dataset instead")
    X_train, y_train, X_test, y_test, class_names = create_synthetic_dataset()

    # Create datasets manually
    train_dataset = TimeSeriesDataset(X_train, y_train, normalize=True)
    test_dataset = TimeSeriesDataset(X_test, y_test, normalize=True)

    X_train_shape = X_train.shape
    X_test_shape = X_test.shape
    unique_labels = sorted(set(y_train))

    dataset_name = "Synthetic"

print(f"Dataset: {dataset_name}")
print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
print(f"Data shape: {X_train_shape}")
print(f"Number of classes: {len(unique_labels)}")

Let's visualize some examples from each class:

In [ ]:
# Plot examples from each class
num_classes = len(unique_labels)
fig, axes = plt.subplots(num_classes, 2, figsize=(12, 3 * num_classes))

for i, cls in enumerate(unique_labels):
    # Find examples from this class
    indices = [idx for idx, (_, y, _) in enumerate(train_dataset) if y == cls]

    if len(indices) == 0:
        continue

    if num_classes == 1:
        ax_row = axes
    else:
        ax_row = axes[i]

    # Plot two examples
    for j in range(min(2, len(indices))):
        idx = indices[j]
        x, _, _ = train_dataset[idx]
        ax = ax_row[j] if num_classes > 1 else ax_row
        ax.plot(x[0].numpy())
        ax.set_title(f"Class {cls}")
        ax.set_xlabel("Time")
        ax.set_ylabel("Value")

plt.tight_layout()
plt.show()

Now, let's create PyTorch dataloaders using our updated loader function:

In [ ]:
# Create dataloaders with validation split
batch_size = 32
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset,
    test_dataset,
    batch_size=batch_size,
    val_split=0.2,
    num_workers=0,
    pin_memory=True,
    shuffle_train=True,
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

## 3. Model Building

Now let's define and configure the ConvTran model using the model factory:

In [ ]:
# Get a sample to determine sequence properties
x_sample, _, _ = train_dataset[0]
seq_len = x_sample.shape[1]  # Length of the sequence
num_channels = x_sample.shape[0]  # Number of channels
num_classes = len(unique_labels)

# Define model parameters
model_config = {
    "Net_Type": "C-T",  # ConvTran
    "Data_shape": [None, num_channels, seq_len],
    "num_labels": num_classes,
    "emb_size": 64,
    "num_heads": 4,
    "dim_ff": 128,
    "dropout": 0.1,
    "Fix_pos_encode": "tAPE",
    "Rel_pos_encode": "None",
}

# Create model using factory function
model = model_factory(model_config)

# Print model summary
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

## 4. Training and Evaluation

Let's set up training using PyTorch:

In [ ]:
# Define training parameters
num_epochs = 30
learning_rate = 1e-3
weight_decay = 1e-4
patience = 10  # For early stopping

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5, verbose=True
)

# Setup for tracking metrics
train_losses = []
val_losses = []
train_accs = []
val_accs = []

# Early stopping variables
best_val_acc = 0
best_model_state = None
epochs_no_improve = 0

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

In [ ]:
# Training function
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:
        x, y = batch[:2]  # First two elements are input and target, third is index
        x, y = x.to(device), y.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward + backward + optimize
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * x.size(0)
        _, predicted = torch.max(outputs, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


# Validation function
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            x, y = batch[:2]  # First two elements are input and target, third is index
            x, y = x.to(device), y.to(device)

            # Forward pass
            outputs = model(x)
            loss = criterion(outputs, y)

            # Statistics
            running_loss += loss.item() * x.size(0)
            _, predicted = torch.max(outputs, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
# Train the model
start_time = time.time()

for epoch in range(num_epochs):
    # Training phase
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Validation phase
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Update learning rate
    scheduler.step(val_loss)

    # Print stats
    print(
        f"Epoch {epoch + 1}/{num_epochs} - "
        f"Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f} - "
        f"Val loss: {val_loss:.4f}, Val acc: {val_acc:.4f}"
    )

    # Check for improvement
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    # Early stopping
    if epochs_no_improve >= patience:
        print(f"Early stopping triggered after {epoch + 1} epochs!")
        break

# Load best model
if best_model_state:
    model.load_state_dict(best_model_state)

training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")

Let's visualize the training process:

In [ ]:
# Plot the learning curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label="Train")
plt.plot(val_accs, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curves")
plt.legend()

plt.tight_layout()
plt.show()

Now let's evaluate the model on the test set:

In [ ]:
# Test the model
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

In [ ]:
# Generate predictions and confusion matrix
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
 for batch in test_loader:
 x, y = batch[:2] # Ignore sample ID if present
 x, y = x.to(device), y.to(device)
 outputs = model(x)
 _, predicted = torch.max(outputs, 1)
 all_labels.extend(y.cpu().numpy())
 all_preds.extend(predicted.cpu().numpy())

# Calculate confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

# Print classification report
print("\nClassification Report:")
print(
 classification_report(
 all_labels, all_preds, target_names=class_names
 )
)

Visualize the confusion matrix:

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 5. Model Analysis

Let's examine the model's attention patterns to understand what it's learning:

In [ ]:
def analyze_attention(model, input_data, sample_idx=0):
    """Analyze the attention patterns of the model"""
    # Get a single input sample
    if isinstance(input_data, torch.utils.data.DataLoader):
        batch = next(iter(input_data))
        x = batch[0][sample_idx : sample_idx + 1]
    else:
        x = input_data[sample_idx : sample_idx + 1]

    model.eval()
    with torch.no_grad():
        # Extract the original input signal
        orig_signal = x[0, 0].cpu().numpy()

    # Add hooks to capture attention weights
    attention_weights = []

    def hook_fn(module, input, output):
        # This assumes the attention module performs matmul(softmax(QK^T/sqrt(d)), V)
        # where the softmax output is the attention weight matrix
        q = module.query(input[0])
        k = module.key(input[0])
        q = q.reshape(q.shape[0], q.shape[1], module.num_heads, -1).transpose(1, 2)
        k = k.reshape(k.shape[0], k.shape[1], module.num_heads, -1).permute(0, 2, 3, 1)
        attn = torch.matmul(q, k) * module.scale
        attn = torch.nn.functional.softmax(attn, dim=-1)
        attention_weights.append(attn.cpu().detach())

        # Register hook
        hook = model.attention_layer.register_forward_hook(hook_fn)

        # Forward pass
        _ = model(x.to(device))

        # Remove hook
        hook.remove()

        return {
            "input_signal": orig_signal,
            "attention_weights": attention_weights[0][0],  # First batch, all heads
        }

In [ ]:
# Analyze attention for a test example
test_batch = next(iter(test_loader))
attention_data = analyze_attention(model, test_batch[0], sample_idx=0)

# Visualize attention weights
attention_weights = attention_data["attention_weights"]
input_signal = attention_data["input_signal"]

print(f"Attention weights shape: {attention_weights.shape}")

# Plot input signal
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(input_signal)
plt.title("Input Signal")
plt.xlabel("Time")
plt.ylabel("Value")

# Plot attention heatmap for the first attention head
plt.subplot(1, 2, 2)
sns.heatmap(attention_weights[0].numpy(), cmap="viridis")
plt.title("Attention Weights (First Head)")
plt.xlabel("Target Position")
plt.ylabel("Source Position")

plt.tight_layout()
plt.show()

Let's also create a CasualConvTran model to compare with the standard ConvTran:

In [ ]:
# Create CasualConvTran model using the model factory
casual_model_config = {
    "Net_Type": "CC-T",  # CasualConvTran
    "Data_shape": [None, num_channels, seq_len],
    "num_labels": num_classes,
    "emb_size": 64,
    "num_heads": 4,
    "dim_ff": 128,
    "dropout": 0.1,
    "Fix_pos_encode": "tAPE",
    "Rel_pos_encode": "None",
}

casual_model = model_factory(casual_model_config)

print("CasualConvTran parameters:", sum(p.numel() for p in casual_model.parameters()))

## 6. Trying Different Positional Encodings

Let's compare different positional encoding strategies:

In [ ]:
# Define models with different positional encodings
models = {
    "No Positional Encoding": ConvTran(
        seq_len=seq_len,
        num_classes=num_classes,
        num_channels=num_channels,
        fix_pos_encode="None",
        rel_pos_encode="None",
    ),
    "Standard Sinusoidal": ConvTran(
        seq_len=seq_len,
        num_classes=num_classes,
        num_channels=num_channels,
        fix_pos_encode="Sin",
        rel_pos_encode="None",
    ),
    "tAPE (Modified Sinusoidal)": ConvTran(
        seq_len=seq_len,
        num_classes=num_classes,
        num_channels=num_channels,
        fix_pos_encode="tAPE",
        rel_pos_encode="None",
    ),
    "Learnable": ConvTran(
        seq_len=seq_len,
        num_classes=num_classes,
        num_channels=num_channels,
        fix_pos_encode="Learn",
        rel_pos_encode="None",
    ),
    "Relative Positional (Scalar)": ConvTran(
        seq_len=seq_len,
        num_classes=num_classes,
        num_channels=num_channels,
        fix_pos_encode="None",
        rel_pos_encode="eRPE",
    ),
}

# Print parameter counts
for name, model in models.items():
    print(
        f"{name}: {sum(p.numel() for p in model.parameters() if p.requires_grad)} parameters"
    )

## 7. Conclusion

In this notebook, we've demonstrated how to use the ConvTran model for time series classification. We've covered:

1. Data preparation and visualization
2. Model configuration with different architectural choices
3. Training and evaluation using PyTorch Lightning
4. Attention pattern visualization and analysis
5. Comparison of different positional encoding strategies

ConvTran combines the advantages of convolutional neural networks for feature extraction with self-attention mechanisms for capturing long-range dependencies, making it a powerful tool for time series classification tasks.